In [1]:
%pip install -q google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\mouza\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key here and press Enter: ")
print("Key saved for this session. Length:", len(os.environ["GEMINI_API_KEY"]))

Paste your Gemini API key here and press Enter:  ········


Key saved for this session. Length: 53


In [3]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.8-flash"   # the model name shown in AI Studio

r = client.models.generate_content(model=MODEL, contents="Say hello in one short sentence.")
print(r.text)

Hello, I hope you are having a wonderful day!


In [4]:
from google.genai import types

tool = types.Tool(function_declarations=[{
    "name": "get_firm_history",
    "description": "Return every fiscal year on record for one firm, oldest first, up to through_fyear.",
    "parameters": {
        "type": "object",
        "properties": {
            "gvkey": {"type": "string", "description": "Six-character Compustat firm id, keep leading zeros, e.g. '001004'"},
            "through_fyear": {"type": "integer", "description": "Last fiscal year the agent may see"},
        },
        "required": ["gvkey"],
    },
}])

config = types.GenerateContentConfig(
    tools=[tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

prompt = ("You are a risk analyst. Assess internal-control risk for the firm with gvkey 001004 "
          "in fiscal year 2018. Start by retrieving the firm's history.")

r = client.models.generate_content(model=MODEL, contents=prompt, config=config)

if r.function_calls:
    call = r.function_calls[0]
    print("Tool called:", call.name)
    print("Arguments:", dict(call.args))
    print("RESULT:", "PASS" if call.name == "get_firm_history" and str(call.args.get("gvkey")) == "001004" else "CHECK")
else:
    print("RESULT: CHECK - model answered without calling the tool")
    print(r.text)

Tool called: get_firm_history
Arguments: {'gvkey': '001004', 'through_fyear': 2018}
RESULT: PASS
